<a href="https://colab.research.google.com/github/natalianowak1/airbnb-price-optimization/blob/main/airbnb_price_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Przygotowanie środowiska i import bibliotek



In [59]:
import pandas as pd

# 2. Wczytanie i wstępny przegląd danych

In [60]:
df = pd.read_excel("/content/Airbnb_Open_Data.xlsx")

Podgląd pierwszych pięciu wierszy

In [61]:
pd.set_option('display.max_columns', None)
print(df.head())

        id                                              NAME      host id  \
0  1001254                Clean & quiet apt home by the park  80014485718   
1  1002102                             Skylit Midtown Castle  52335172823   
2  1002403               THE VILLAGE OF HARLEM....NEW YORK !  78829239556   
3  1002755                                               NaN  85098326012   
4  1003689  Entire Apt: Spacious Studio/Loft by central park  92037596077   

  host_identity_verified host name neighbourhood group neighbourhood  \
0            unconfirmed  Madaline            Brooklyn    Kensington   
1               verified     Jenna           Manhattan       Midtown   
2                    NaN     Elise           Manhattan        Harlem   
3            unconfirmed     Garry            Brooklyn  Clinton Hill   
4               verified    Lyndon           Manhattan   East Harlem   

   latitude  longitude        country country code  instant_bookable  \
0  40.64749  -73.97237  United S

Identyfikacja zmiennych i weryfikacja typów

In [62]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   latitude                        102591 non-null  float64
 8   longitude                       102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  float64
 12  cancellation_pol

Zliczanie brakujących wartości (nulli) w każdej kolumnie

In [63]:
print(df.isnull().sum())

id                                     0
NAME                                 250
host id                                0
host_identity_verified               289
host name                            406
neighbourhood group                   29
neighbourhood                         16
latitude                               8
longitude                              8
country                              532
country code                         131
instant_bookable                     105
cancellation_policy                   76
room type                              0
Construction year                    214
price                                247
service fee                          247
minimum nights                       409
number of reviews                    183
reviews per month                  15879
review rate number                   326
calculated host listings count       319
availability 365                     448
house_rules                        52131
license         

Liczba zduplikowanych wierszy

In [64]:
print(df.duplicated().sum())

541


Wnioski ze wstępnej analizy:
1. Metryki ogólne:
*  Tabela zawiera 25 kolumnn o nazwach kolejno: 'id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'license'
* Tabela liczy 102 599 wierszy (ogłoszeń).
* W zbiorze wykryto 541 całkowicie zduplikowanych wierszy, które należy usunąć.
* Jedyne kolumny, które są w 100% kompletne (0 nulli), to: `id`, `host id` oraz `room type`. W pozostałych 22 kolumnach występują nulle.



2. Zidentyfikowane anomalie i plan ich czyszczenia:
* Zbiór danych charakteryzuje się brakiem spójności w nazwach kolumn (część pisana wielkimi literami, stosowanie spacji na zmianę z podkreśleniami/podłogami). Aby uprosćić kod w Pythonie (zamiast konieczności pisania "df['host id']", można pisać: "df.host_id") oraz zapewnić spójność, nazwy kolumn należy zamienić do formatu `pierwszesłowo_drugiesłowo` (małe litery, spacje zastąpione znakiem `_`).
*   Kolumny takie jak 'price' (cena) oraz 'service fee' (opłata serwisowa) są już zapisane jako dane liczbowe (float64), co umożliwia bezpośrednie wykonywanie obliczeń i analizę statystyczną. Posiadają jednak po 247 braków danych, które trzeba uzupełnić (np. medianą cen).
* Kolumna `instant_bookable` zawiera wartości binarne (1 i 0) oraz 105 nulli. W etapie czyszczenia należy zastąpić nulle wartością `0` (bezpieczne założenie, że brak informacji oznacza brak natychmiastowej rezerwacji), a całą kolumnę zmienić na typ logiczny (`boolean`).
* Kolumna `reviews per month` zawiera aż 15 879 braków danych. Wynika to z faktu, że nowo dodane nieruchomości nie mają jeszcze żadnych opinii. Nulle w tej kolumnie zostaną zastąpione wartością `0.0`.
* Kolumna 'license' zawiera tylko 2 wypełnione wiersze, a cała reszta to nulle (kolumna kwalifikuje się do usunięcia).
* Kolumna `house_rules` zawiera informacje w mniej niż połowie wierszy (52 131 nulli). Braki zostaną zastąpione wartością domyślną "No rules specified".
* Należy wykonać korektę kolumn lokalizacyjnych (`country` i `country code`) Ponieważ cały zbiór danych dotyczy rynku w Nowym Jorku, braki danych w kolumnie kraju i jego kodu zostaną uzupełnione stałymi wartościami (odpowiednio: "United States" oraz "US")
* Kolumna Zależność weryfikacji hosta (`host_identity_verified`) posiada 289 nulli. Zostanie tam wprowadzona kategoria "unconfirmed", co jest bezpieczniejszym podejściem biznesowym niż zakładanie, że profil jest zweryfikowany
* Wykryto po 8 braków danych w pozycjach geograficznych (`latitude`, `longitude`). Ponieważ te współrzędne są kluczowe do poprawnego renderowania map w Tableau, wiersze z tymi brakami zostaną całkowicie usunięte z bazy.
* Pozostałe kolumny tekstowe i lokalizacyjne (np. `NAME`, `host name`, `neighbourhood`) posiadają niewielkie liczby braków (od kilku do kilkuset). W ich przypadku wiersze z nullami zostaną zastąpione wartością "Unknown".



# 3. Czyszczenie danych (Data Cleaning)


Usuwanie całkowicie zduplikowanych wierszy

In [65]:
print(f"Liczba wierszy przed usunięciem duplikatów: {len(df)}")
df = df.drop_duplicates()
print(f"Liczba wierszy po usunięciu duplikatów: {len(df)}\n")

Liczba wierszy przed usunięciem duplikatów: 102599
Liczba wierszy po usunięciu duplikatów: 102058



Usuwanie wierszy, które mają nulle w latitude lub longitude

In [66]:
print(f"Liczba wierszy przed usunięciem braków w geolokalizacji: {len(df)}")
df = df[(df['latitude'].notnull()) & (df['longitude'].notnull())]
print(f"Liczba wierszy po usunięciu braków w geolokalizacji: {len(df)}")

Liczba wierszy przed usunięciem braków w geolokalizacji: 102058
Liczba wierszy po usunięciu braków w geolokalizacji: 102050


Usuwanie kolumny `license`

In [67]:
df = df.drop(columns=['license'])
print(f"Nowa liczba kolumn w tabeli: {len(df.columns)}")

Nowa liczba kolumn w tabeli: 24


Zmiana nazw kolumn do formatu "snake_case"

In [68]:
print(list(df.columns))
#zamiana liter z wielkich ma małe
df.columns = df.columns.str.lower()
#zamiana spacji w nazwach kolumn na podkreśclenie
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(list(df.columns))

['id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules']
['id', 'name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'house_rules']


Uzupełnienie Nulli zerami w kolumnie `instant_bookable`

In [69]:
print(f"Liczba nulli w instant_bookable przed uzupełnieniem zerami: {df['instant_bookable'].isnull().sum()}")
df['instant_bookable'] = df['instant_bookable'].fillna(0)
print(f"Liczba nulli w instant_bookable: {df.instant_bookable.isnull().sum()}")

Liczba nulli w instant_bookable przed: 105
Liczba nulli w instant_bookable: 0


Zmieniamy typ kolumny `instant_bookable` na boolean

In [70]:
print(f"Typ kolumny przed zmianą: {df.instant_bookable.dtype}")
df['instant_bookable'] = df['instant_bookable'].astype(bool)
print(f"Nowy typ kolumny: {df.instant_bookable.dtype}")

Typ kolumny: float64
Nowy typ kolumny: bool


Uzupełnienie nulli w recenzjach na miesiąc (`reviews_per_month`) zerem

In [71]:
print(f"Liczba nulli w reviews per month przed uzupełnieniem: {df.reviews_per_month.isnull().sum()}")
df['reviews_per_month'] = df['reviews_per_month'].fillna(0.0)
print(f"Liczba nulli w reviews per month: {df.reviews_per_month.isnull().sum()}")

Liczba nulli w reviews per month przed: 15817

Liczba nulli w reviews per month: 0


Zastępowanie braków tekstem domyślnym ("No rules specified") w kolumnie `house_rules`

In [72]:
print(f"Liczba nulli w house_rules przed uzupełnieniem: {df.house_rules.isnull().sum()}\n")
df['house_rules'] = df['house_rules'].fillna("No rules specified")
print(f"Liczba nulli w house_rules: {df.house_rules.isnull().sum()}\n")

Liczba nulli w house_rules przed: 51840

Liczba nulli w house_rules: 0



Uzupełnianie stałą wartością kolumny `country`

In [73]:
print(f"Liczba nulli w country przed uzupełnieniem: {df.country.isnull().sum()}")
df['country'] = df['country'].fillna("United States")
print(f"Liczba nulli w country: {df.country.isnull().sum()}")

Liczba nulli w country przed: 532
Liczba nulli w country: 0


Uzupełnianie stałą wartością kolumny `country_code`

In [74]:
print(f"Liczba nulli w country_code przed uzupełnieniem: {df.country_code.isnull().sum()}")
df['country_code'] = df['country_code'].fillna("US")
print(f"Liczba nulli w country_code: {df.country_code.isnull().sum()}")

Liczba nulli w country_code przed: 131
Liczba nulli w country_code: 0


Weryfikacja gospodarza `host_identity_verified`: uzupełnienie nulli wartością "unconfirmed"

In [75]:
print(f"Liczba nulli w host_identity_verified przed uzupełnieniem: {df.host_identity_verified.isnull().sum()}\n")
df['host_identity_verified'] = df['host_identity_verified'].fillna("unconfirmed")
print(f"Liczba nulli w host_identity_verified: {df.host_identity_verified.isnull().sum()}")

Liczba nulli w host_identity_verified przed: 289

Liczba nulli w host_identity_verified: 0


Zastępowanie braków w kolumnach `name` i `host_name` wartością "Unknown"

In [76]:
print(f"Liczba nulli w name przed uzupełnieniem: {df.name.isnull().sum()}")
print(f"Liczba nulli w host_name przed uzupełnieniem: {df.host_name.isnull().sum()}\n")
df['name'] = df['name'].fillna("Unknown")
df['host_name'] = df['host_name'].fillna("Unknown")
print(f"Liczba nulli w name: {df.name.isnull().sum()}")
print(f"Liczba nulli w host_name: {df.host_name.isnull().sum()}")

Liczba nulli w name przed: 250
Liczba nulli w host_name przed: 404

Liczba nulli w name: 0
Liczba nulli w host_name: 0
